In [3]:
from pathlib import Path
import os
ROOT = Path().resolve()
print("Project root:", ROOT)
os.chdir(Path(os.getcwd()).parent)  # von /notebooks nach Projekt-Root springen
print("Working dir:", Path().absolute())

Project root: C:\Users\maumo\OneDrive\Dokumente\rag_project
Working dir: c:\Users\maumo\OneDrive\Dokumente


In [ ]:
from processing import read_pdf, chunk_documents  # evtl. später für Tests
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama

# Embeddings wieder initialisieren (gleich wie in 02_build_index)
embedder = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)
print("✅ Embedder initialisiert: BAAI/bge-m3")

# Pfad zum bestehenden Chroma-Index
CHROMA_DIR = ROOT / "artifacts" / "chroma_din"
print("Lade Vectorstore aus:", CHROMA_DIR)

vs = Chroma(
    embedding_function=embedder,
    persist_directory=str(CHROMA_DIR),
)

retriever = vs.as_retriever(search_kwargs={"k": 4})
print("✅ Vectorstore geladen & Retriever initialisiert.")


✅ Embedder initialisiert: BAAI/bge-m3
Lade Vectorstore aus: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts\chroma_din
✅ Vectorstore geladen & Retriever initialisiert.


In [4]:
def answer_with_rag(
    query: str,
    vs,
    model_name: str = "llama3.1:8b",  # oder "llama4:8b" – was du in Ollama hast
    k: int = 4,
):
    """
    Einfacher RAG-Helper:
    - holt k relevante Chunks aus dem Vectorstore
    - baut einen Kontext
    - ruft ein lokales LLM (Ollama) auf
    - gibt Antwort + Quellen zurück
    """
    retriever = vs.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query)

    # Kontext aus den gefundenen Chunks bauen
    context_blocks = []
    for d in docs:
        src = d.metadata.get("source")
        page = d.metadata.get("page")
        context_blocks.append(
            f"[Quelle: {src}, Seite {page}]\n{d.page_content}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
Du bist ein Assistent für Fragen zu technischen Normen.
Nutze ausschließlich den bereitgestellten Kontext, um die Frage zu beantworten.
Wenn etwas nicht im Kontext steht, sage ehrlich, dass die Information nicht vorliegt.

KONTEXT:
{context}

FRAGE:
{query}

ANTWORT (auf Deutsch, kurz und präzise):
"""

    llm = ChatOllama(model=model_name, temperature=0.1)
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "sources": [
            {
                "source": d.metadata.get("source"),
                "page": d.metadata.get("page"),
                "filepath": d.metadata.get("filepath"),
            }
            for d in docs
        ],
    }


In [12]:
from pathlib import Path

ROOT = Path().resolve()
ARTIFACTS_DIR = ROOT / "artifacts"
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("Existiert:", ARTIFACTS_DIR.exists())

if ARTIFACTS_DIR.exists():
    for sub in ARTIFACTS_DIR.iterdir():
        print("Unterordner:", sub)
        if sub.is_dir():
            for f in sub.iterdir():
                print("   -", f.name)


ARTIFACTS_DIR: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts
Existiert: True
Unterordner: C:\Users\maumo\OneDrive\Dokumente\rag_project\artifacts\chroma_din
   - bd1abfdc-bac6-47f7-af1f-dead87ec0493
   - chroma.sqlite3


In [20]:
query = "Was ist ein Ferrari?"

retriever = vs.as_retriever(search_kwargs={"k": 4})
docs = retriever.invoke(query)

print("Anzahl gefundener Dokumente:", len(docs))
for i, d in enumerate(docs):
    print(f"\n--- Doc {i} ---")
    print("Source:", d.metadata.get("source"))
    print("Page:", d.metadata.get("page"))
    print(d.page_content[:400], "...")


Anzahl gefundener Dokumente: 4

--- Doc 0 ---
Source: din_5566_2.pdf
Page: 2
11  6.2.2 Ergänzende Festlegungen ................................................................................................................................... 11  6.3 Bedienelemente ..................................................................................................................................................... 12  6.4 Weitere Ausstattung für den Führerraum............... ...

--- Doc 1 ---
Source: din_5566_2.pdf
Page: 4
nur die in Bezug genommene Ausgabe. Bei undatierten Verweisungen gilt die letzte  Ausgabe des in Bezug genommenen Dokuments (einschließlich aller Änderungen).  DIN 5566-1:2020-05, Schienenfahrzeuge — Führerräume — Teil 1: Allgemeine Anforderungen  DIN EN 12663-1, Bahnanwendungen — Festigkeitsanforderungen an Wagenkästen von Schienenfahrzeugen —  Teil 1: Lokomotiven und Personenfahrzeuge (und alter ...

--- Doc 2 ---
Source: din_5566_2.pdf
Page: 14
DIN 5566-2:2020-05  

In [21]:
query = "Was ist ein Ferrari?"
rag_result = answer_with_rag(query, vs)

print("Frage:", query)
print("\nAntwort:\n", rag_result["answer"])
print("\nQuellen:")
for s in rag_result["sources"]:
    print(f"- {s['source']} (Seite {s['page']}) → {s['filepath']}")


Frage: Was ist ein Ferrari?

Antwort:
 Ich kann diese Frage nicht beantworten. Der bereitgestellte Kontext beschäftigt sich ausschließlich mit technischen Normen für Führerräume von Eisenbahnfahrzeugen und enthält keine Informationen über Ferrari.

Quellen:
- din_5566_2.pdf (Seite 2) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 4) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 14) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf
- din_5566_2.pdf (Seite 12) → C:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\din_5566_2.pdf


In [34]:
from src.indexing import collect_documents
from collections import Counter

docs = collect_documents()
print("Anzahl Seiten insgesamt:", len(docs))
print("Normen-Verteilung (vor dem Indexing):")
print(Counter(d.metadata["source"] for d in docs))


Anzahl Seiten insgesamt: 75
Normen-Verteilung (vor dem Indexing):
Counter({'din_en_12663_1.pdf': 40, 'din_5566_1.pdf': 18, 'din_5566_2.pdf': 17})


In [31]:
from src.indexing import load_index
from src.rag import answer_with_rag, debug_retrieval

vs = load_index()

query = "Welche Festigkeitsanforderungen an Wagenkästen von Schienenfahrzeugen gibt es?"

# Debug (optional):
debug_retrieval(query, vs, k=3)

# RAG-Antwort:
#res = answer_with_rag(query, vs)
#print("Antwort:\n", res["answer"])
#print("\nQuellen:")
#for s in res["sources"]:
#    print(f"- {s['source']} (Seite {s['page']}) → {s['filepath']}")


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 120ad235-0d5d-4bec-8454-ee1074be5aa5)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Anzahl gefundener Dokumente: 3

--- Doc 0 ---
Source: din_5566_2.pdf
Page: 4
nur die in Bezug genommene Ausgabe. Bei undatierten Verweisungen gilt die letzte  Ausgabe des in Bezug genommenen Dokuments (einschließlich aller Änderungen).  DIN 5566-1:2020-05, Schienenfahrzeuge — Führerräume — Teil 1: Allgemeine Anforderungen  DIN EN 12663-1, Bahnanwendungen — Festigkeitsanforderungen an Wagenkästen von Schienenfahrzeugen —  Teil 1: Lokomotiven und Personenfahrzeuge (und alter ...

--- Doc 1 ---
Source: din_5566_2.pdf
Page: 17
DIN 5566-2:2020-05  17  UIC 553:2003, Lüftung, Heizung und Klimatisierung der Reisezugwagen4  UIC 566:1990, Beanspruchungen von Reisezugwagenkästen und deren Anbauteilen4  UIC 651:2002, Gestaltung der Führerräume von Lokomotiven, Triebwagen, Triebwagenzügen und  Steuerwagen4  VDI 2057 Blatt 1, Einwirkung mechanischer Schwingungen auf den Menschen — Ganzkörper-Schwingungen  VDI 2057 Blatt 2, Einwir ...

--- Doc 2 ---
Source: din_5566_2.pdf
Page: 2
7  5.2.2 Fluchtweg u

Hybrid Testing für RAG Prompts

In [6]:
import os

os.chdir(r"C:\Users\maumo\OneDrive\Dokumente\rag_project")
print("Working dir:", os.getcwd())



Working dir: C:\Users\maumo\OneDrive\Dokumente\rag_project


In [8]:
from src.rag import answer_with_rag_hybrid

query = "Welche Anforderungen gelten für die statischen Werkstoffkennwerte?"

result = answer_with_rag_hybrid(
    query=query,
    k=5
)

print("Antwort:\n", result["answer"])
print("\nQuellen:")
for s in result["sources"]:
    print(f"- {s['source']} (Seite {s['page']})")


Antwort:
 Die begrenzenden statischen Werkstoffkennwerte müssen den minimalen Dehngrenzen/Streckgrenzen und Zugfestigkeiten der Werkstoff-Spezifikationen entsprechen. Die verwendeten Werte sollten den entsprechenden europäischen, internationalen oder nationalen Normen entnommen werden. Sind solche Normen nicht vorhanden, müssen die am besten geeigneten alternativen Datenquellen verwendet werden. Die statischen Werkstoffkennwerte müssen konsistent mit den Daten in europäischen oder nationalen Werkstoffnormen sein.

Quellen:
- din_en_12663_1.pdf (Seite 28)
- din_5566_2.pdf (Seite 4)
- din_en_12663_1.pdf (Seite 11)
- din_5566_2.pdf (Seite 8)
- din_en_12663_1.pdf (Seite 9)
